# Week 8-1 · Tutorial — Doubt-Solving on Python after DMP-01
**Module:** Data Analysis & Modeling in Python (tutorial) · Instructor: Manusha Rao.

A hands-on clinic reinforcing DMP-01: **fetching data** (yfinance gotchas + CSV round-trip), **time zones**,
**vectorization vs iteration**, and a full **moving-average crossover back-test** (SMA 21 / LMA 63).

> The live parts use `yfinance` (not installed here), so we work from the **shipped NIFTY-50 daily file**
> `nifty50_daily.csv` (962 bars, 2017 → 2020). The yfinance-specific points are explained and shown as the
> exact code you'd run.

## 1. The core libraries (one line each)
| Library | Used for |
|---|---|
| `pandas` | data analysis & manipulation (DataFrames) |
| `numpy` | arrays, linear algebra, vectorized maths |
| `datetime` | building/​manipulating dates & times |
| `yfinance` | downloading OHLCV data for tickers |
| `warnings` | silence non-fatal warning messages |
| `matplotlib` | plotting & visualization |
| `cufflinks` | bridge pandas → Plotly for interactive `.iplot()` charts |

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print("pandas", pd.__version__)

pandas 3.0.3


## 2. A date is not a string, and not a tuple
`start = (2017, 1, 1)` is a **tuple** (round brackets); `"2017-01-01"` is a **string**. `yfinance.download`
needs real **date** objects, so passing a tuple raises *`'tuple' object has no attribute 'timestamp'`*.
Build dates with `datetime`.

In [2]:
maybe_tuple = (2017, 1, 1)
print("type of (2017,1,1):", type(maybe_tuple).__name__)        # tuple
print("type of '2017-01-01':", type("2017-01-01").__name__)      # str
start_date = dt.date(2017, 1, 1)
end_date   = dt.date(2020, 11, 27)
print("type of dt.date(...):", type(start_date).__name__, "->", start_date)

type of (2017,1,1): tuple
type of '2017-01-01': str
type of dt.date(...): date -> 2017-01-01


### The yfinance call (for reference)
Two arguments matter a lot:
- `multi_level_index=False` — flattens the new multi-level columns (no `('Close','^NSEI')` mess).
- `auto_adjust` — **True** (default) gives corporate-action-adjusted prices; **False** adds a separate
  `Adj Close` column and leaves OHLC unadjusted.
```python
import yfinance as yf
df = yf.download("^NSEI", start=start_date, end=end_date,
                 multi_level_index=False, auto_adjust=True)
# intraday (note: Yahoo only serves the last 60 days for intraday):
intraday = yf.download("^NSEI", period="60d", interval="5m",
                       multi_level_index=False)
```

## 3. Load the shipped CSV the right way
We read the same NIFTY data from disk. First the *wrong* way to expose the defaults, then the clean way.

In [3]:
# plain read: Date is just a column, index is a RangeIndex
raw = pd.read_csv("nifty50_daily.csv")
print("plain read -> index:", type(raw.index).__name__, "| columns:", list(raw.columns))
raw.info()

plain read -> index: RangeIndex | columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
<class 'pandas.DataFrame'>
RangeIndex: 962 entries, 0 to 961
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    962 non-null    str    
 1   Open    962 non-null    float64
 2   High    962 non-null    float64
 3   Low     962 non-null    float64
 4   Close   962 non-null    float64
 5   Volume  962 non-null    int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 45.2 KB


### `index_col` + `parse_dates`
`index_col=0` (or `'Date'`) makes Date the index; `parse_dates=True` turns the date strings into a real
`DatetimeIndex`. Either the column **number** or its **name** works.

In [4]:
df = pd.read_csv("nifty50_daily.csv", index_col="Date", parse_dates=True)
print("clean read -> index:", type(df.index).__name__)
print("shape:", df.shape, "| df.shape[0] rows =", df.shape[0], ", df.shape[1] cols =", df.shape[1])
print(df.head(3))

clean read -> index: DatetimeIndex
shape: (962, 5) | df.shape[0] rows = 962 , df.shape[1] cols = 5
                   Open         High          Low    Close  Volume
Date                                                              
2017-01-02  8210.099609  8212.000000  8133.799805  8179.50  118300
2017-01-03  8196.049805  8219.099609  8148.600098  8192.25  127300
2017-01-04  8202.650391  8218.500000  8180.899902  8190.50  132400


### Saving back out — `to_csv`
Round-tripping to disk is the lecture's contingency tip: download once, save, reload offline.
```python
df.to_csv("my_data.csv")     # any name you like
```

In [5]:
df.to_csv("_roundtrip_check.csv")
back = pd.read_csv("_roundtrip_check.csv", index_col="Date", parse_dates=True)
import os; os.remove("_roundtrip_check.csv")
print("round-trip identical shape:", back.shape == df.shape)

round-trip identical shape: True


### Windows file paths
A backslash starts an escape sequence, so a raw Windows path breaks. Use a **raw string** `r"..."` or
**double backslashes**:
```python
pd.read_csv(r"C:\Users\me\Downloads\data.csv")   # raw string
pd.read_csv("C:\\Users\\me\\Downloads\\data.csv") # doubled
```

## 4. Adjusted vs unadjusted close
`Adj Close` bakes in **corporate actions** — dividends, stock splits, issuances, mergers — applied
*backwards* over history so returns are continuous. For an **index** like NIFTY-50 there are no splits, so
Adj Close ≈ Close. For long-horizon analysis, prefer **adjusted** close.

## 5. Time zones
yfinance intraday timestamps arrive in **UTC** (`+00:00`). Convert with `tz_convert`; list zones via `pytz`
(or the stdlib `zoneinfo`, used here).
```python
intraday.index = intraday.index.tz_convert("Asia/Kolkata")  # IST
import pytz; pytz.all_timezones[:3]
```

In [6]:
from zoneinfo import available_timezones
zones = available_timezones()
print("total tz names:", len(zones))
print("a few:", sorted(z for z in zones if "Kolkata" in z or "London" in z)[:3])
# demonstrate a UTC -> IST conversion on a sample timestamp index
idx = pd.date_range("2020-11-27 09:30", periods=3, freq="5min", tz="UTC")
print("UTC :", [t.strftime("%H:%M") for t in idx])
print("IST :", [t.strftime("%H:%M") for t in idx.tz_convert("Asia/Kolkata")])

total tz names: 598
a few: ['Asia/Kolkata', 'Europe/London']
UTC : ['09:30', '09:35', '09:40']
IST : ['15:00', '15:05', '15:10']


## 6. Vectorization vs iteration (the event-driven analogy)
A **list** `* 3` *repeats* the list; a **numpy array** `* 3` multiplies every element **at once**
(vectorized). Doing it element-by-element needs a `for` loop — the same shape as **event-driven** back-testing.

In [7]:
number_list = [1, 2, 11, 5, 8]
print("list * 3  (repeat):", number_list * 3)
arr = np.array(number_list)
print("array * 3 (vectorized):", arr * 3)
loop = [v * 3 for v in number_list]        # event-driven style: one element at a time
print("for-loop  (iterated):", loop)

list * 3  (repeat): [1, 2, 11, 5, 8, 1, 2, 11, 5, 8, 1, 2, 11, 5, 8]
array * 3 (vectorized): [ 3  6 33 15 24]
for-loop  (iterated): [3, 6, 33, 15, 24]


## 7. The moving-average crossover back-test
**Rule:** go **long** when the short MA crosses **above** the long MA; go **short/flat** when it crosses below.
Windows: short `SMA = 21`, long `LMA = 63`. We run all 7 vectorized steps: data → indicators → signal →
position → returns → equity → analysis.

In [8]:
data = pd.read_csv("nifty50_daily.csv", index_col="Date", parse_dates=True)[["Close"]].copy()
SW, LW = 21, 63
data["SMA"] = data["Close"].rolling(SW).mean()
data["LMA"] = data["Close"].rolling(LW).mean()
data = data.dropna()
print("rows after warm-up:", data.shape[0])
print(data.head(2))

rows after warm-up: 900
                  Close          SMA          LMA
Date                                             
2017-04-03  9237.849609  9067.978516  8759.315027
2017-04-05  9265.150391  9085.483305  8776.547573


### Signal on the crossover (today vs yesterday)
Long entry: today `SMA > LMA` **and** yesterday `SMA <= LMA`. We capture the *state* (long when SMA>LMA) and
detect crossings with `.diff()`.

In [9]:
data["regime"] = np.where(data["SMA"] > data["LMA"], 1, -1)   # +1 long, -1 short
data["signal"] = data["regime"].diff()                        # +2 = golden cross, -2 = death cross
crosses = data["signal"].value_counts().to_dict()
print("crossings (2.0=golden up, -2.0=death down):", crosses)

crossings (2.0=golden up, -2.0=death down): {0.0: 891, -2.0: 4, 2.0: 4}


### Position, shift, returns vs buy-and-hold

In [10]:
data["position"]  = data["regime"].shift(1)          # act next bar (no look-ahead)
data["cc"]        = data["Close"].pct_change()
data = data.dropna()
data["strat"]     = data["position"] * data["cc"]
data["bh_eq"]     = (1 + data["cc"]).cumprod()
data["str_eq"]    = (1 + data["strat"]).cumprod()
bh  = data["bh_eq"].iloc[-1] - 1
st  = data["str_eq"].iloc[-1] - 1
print(f"period {data.index[0].date()} -> {data.index[-1].date()}")
print(f"Buy & Hold     : {bh*100:.2f}%")
print(f"SMA21/LMA63 x  : {st*100:.2f}%")
print("time long:", int((data['position']==1).sum()), "| time short:", int((data['position']==-1).sum()))

period 2017-04-05 -> 2020-11-27
Buy & Hold     : 40.39%
SMA21/LMA63 x  : 68.04%
time long: 692 | time short: 207


### Equity curves

In [11]:
fig, ax = plt.subplots(figsize=(8,3.6))
ax.plot(data.index, data["bh_eq"],  label=f"Buy & Hold ({bh*100:.1f}%)", color="#7f1d1d", lw=1.3)
ax.plot(data.index, data["str_eq"], label=f"Crossover ({st*100:.1f}%)", color="#dc2626", lw=1.3)
ax.axhline(1, color="gray", ls="--", lw=0.7)
ax.set_title("NIFTY-50 — Buy & Hold vs SMA21/LMA63 crossover")
ax.set_ylabel("Growth of 1"); ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig("xover_preview.png", dpi=110)
print("equity curve drawn")

equity curve drawn

## Key takeaways
- **Build dates with `datetime`** — a tuple `(2017,1,1)` or a string isn't a date; `yf.download` needs real date objects.
- **yfinance flags:** `multi_level_index=False` (flatten columns), `auto_adjust=True` (corporate-action-adjusted prices; `Adj Close` only appears when `False`). Intraday is limited to the **last 60 days**.
- **CSV round-trip:** `to_csv` to save, `read_csv(index_col=..., parse_dates=True)` to reload with a proper `DatetimeIndex`. Use raw strings `r"..."` for Windows paths.
- **Time zones:** yfinance intraday is UTC; `tz_convert("Asia/Kolkata")` for IST; `pytz.all_timezones` lists them.
- **Vectorization:** array ops run all at once (fast); a `for` loop is element-by-element — the shape of event-driven back-testing.
- **MA crossover** in 7 vectorized steps: indicators (`rolling().mean()`) → regime (`np.where`) → crossing (`.diff()`) → position (`.shift(1)`) → returns (`position * pct_change`) → equity (`cumprod`) → compare to buy-and-hold.